# Insurance Claims Image Similarity with MongoDB Vector Search

[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mongodb-industry-solutions/Insurance-image-search/blob/main/image_similarity.ipynb)

### Install Dependencies
Install and pin the Python packages needed for image processing, model inference, dataset loading, plotting, and MongoDB access.

In [ ]:
%pip install -U -q pymongo pillow===11.3.0 torchvision datasets matplotlib

### Build the Image Vectorizer
Define and initialize the TorchVision-based embedding model that converts each image into a numeric vector.

In [ ]:
from PIL import Image
from torchvision import transforms as ts
import torchvision.models as models

# Definition of the image embedder class
class ImageVectorizer:
    def __init__(self):
        self.normalize = ts.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )
        # https://pytorch.org/vision/stable/models/generated/torchvision.models.squeezenet1_0.html
        weights = models.SqueezeNet1_0_Weights.DEFAULT
        self.model = models.squeezenet1_0(weights=weights, progress=False)
        self.model.eval()

    def vectorize(self, image_file_name):
        image = Image.open(image_file_name).convert('RGB')
        image = ts.Resize(256)(image)
        image = ts.CenterCrop(224)(image)
        tensor = ts.ToTensor()(image)
        tensor = self.normalize(tensor).reshape(1, 3, 224, 224)
        vector = self.model(tensor).cpu().detach().numpy().flatten()
        return vector

image_vectorizer = ImageVectorizer()

### Download and Prepare the Dataset
Fetch the sample vehicle damage dataset and write a local image set used for embedding and similarity search.

In [ ]:
import os
from datasets import load_dataset, features

# Define the features to ensure the 'label' is treated as a string, not an integer class.
# This prevents the 'Invalid string class label' error.
dataset_features = features.Features({
    'image': features.Image(decode=True),
    'label': features.Value('string')
})

# Explicitly specify the split to avoid any ambiguity and load directly into the 'dataset' variable.
dataset = load_dataset('chittaranjankhatua/car_damage_pub', split='train', features=dataset_features)

os.makedirs('car_damage', exist_ok=True)
# The dataset variable now directly holds the 'train' split, so access its 'image' feature directly.
max_dataset_images = int(os.environ.get('NOTEBOOK_MAX_DATASET_IMAGES', len(dataset['image'])))

# Writes dataset to file
for i in range(min(max_dataset_images, len(dataset['image']))):
    image = dataset['image'][i]
    image_name = str('car_damage/')+str(i)+str('.jpg')
    image.save(image_name)

Next, we'll define two functions that will help us with data visualization. The first function, `show_image`, takes an image path as input and displays the image using Matplotlib. The second function, `show_images`, takes a list of image paths and displays them in a grid format.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def display_multiple_images(file_names):
    num_images = len(file_names)
    fig, axes = plt.subplots(1, num_images)
    fig.set_figwidth(1.5 * num_images)
    for ax, file_name in zip(axes, file_names):
        ax.imshow(Image.open(file_name))
        ax.axis('off')
    plt.show()

def display_single_image(file_name):
    fig, ax = plt.subplots(1, 1)
    fig.set_figwidth(1.3)
    ax.imshow(Image.open(file_name))
    ax.axis('off')

### Connect to MongoDB
Initialize the MongoDB client from `MONGODB_URI` and verify connectivity with a ping. If you don't have a configured MongoDB URI, you can register for a free [MongoDB Atlas account](https://www.mongodb.com/cloud/atlas) and create a cluster to obtain your connection string.

In [ ]:
import os
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

# Set MONGODB_URI in your shell before launching Jupyter.
uri = os.environ.get(
    'MONGODB_URI',
    'mongodb+srv://<username>:<password>@<cluster-name>/?retryWrites=true&w=majority',
)

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'), appname='is-insurance-claims-image-vector-search')

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print('Pinged your deployment. You successfully connected to MongoDB!')
except Exception as e:
    print(e)

### Load Images and Embeddings into MongoDB
Vectorize each saved image and insert the filename, binary content and vector embedding into the MongoDB collection.

In [ ]:
from PIL import Image
import os

db = client['claim_resolution']
coll = db['car_damage_photos']

if os.environ.get('NOTEBOOK_CLEAR_COLLECTION', 'false').lower() == 'true':
    coll.delete_many({})

image_folder = 'car_damage'
image_files = [f for f in os.listdir(image_folder) if os.path.isfile(os.path.join(image_folder, f))]

for image_file in image_files:
    # Construct the full path to the image file
    image_path = os.path.join(image_folder, image_file)

    # Read the image from file as binary data
    with open(image_path, 'rb') as image_file_obj:
        image_data = image_file_obj.read()

    img_embedding = image_vectorizer.vectorize(image_path).tolist()

    image_document = {
        'filename': image_file,
        'data': image_data,
        'embedding': img_embedding
    }

    coll.insert_one(image_document)
    print(f'Inserted: {image_file}')

print('All images inserted into MongoDB.')

### Create the Vector Search Index
Create the `image_vector_index` index used for vector similarity queries.

In [ ]:
vector_search_index_definition = {
    'name': 'image_vector_index',
    'type': 'vectorSearch',
    'definition': {
        'fields': [
            {
                'type': 'vector',
                'path': 'embedding',
                'numDimensions': 1000,
                'similarity': 'cosine',
            }
        ]
    },
}

coll.create_search_index(model=vector_search_index_definition)
list(coll.list_search_indexes(name='image_vector_index'))

### Run the Similarity Query
Embed a sammple query image, run `$vectorSearch` and display the top matching images. Feel free to change the query image to any other image URL to see how the results change.

In [ ]:
import os
import requests
import tempfile

db = client['claim_resolution']
coll = db['car_damage_photos']

# Define the source of the query image (could be a URL or a local path)
original_query_image_source = 'https://raw.githubusercontent.com/mongodb-industry-solutions/Insurance-image-search/refs/heads/main/test.jpg'
query_image_path = None # This will store the path to the local image file

# Check if the source is a URL and download it if necessary
if original_query_image_source.startswith('http://') or original_query_image_source.startswith('https://'):
    response = requests.get(original_query_image_source)
    response.raise_for_status()  # Raise an exception for HTTP errors

    # Create a temporary file to save the image
    with tempfile.NamedTemporaryFile(delete=False, suffix='.jpg') as tmp_file:
        tmp_file.write(response.content)
        query_image_path = tmp_file.name
else:
    # If it's a local path, use it directly
    query_image_path = original_query_image_source

image_folder = 'car_damage'

# Use the local path for vectorization
query_embedding = image_vectorizer.vectorize(query_image_path).tolist()

documents = coll.aggregate([
  {
    '$vectorSearch': {
      'index': 'image_vector_index',
      'path': 'embedding',
      'queryVector': query_embedding,
      'numCandidates': 100,
      'limit': 5
    }
  },
  {
    '$project': {
      'filename': 1,
      'score': {'$meta': 'vectorSearchScore'}
    }
  }
])

documents = list(documents)

similar_images_list = []

if not documents:
    print('No similar images found. Please ensure your Vector Search index "image_vector_index" is correctly configured and populated.')
else:
    for i in range(min(5, len(documents))):
        image_file = documents[i]['filename']
        image_path = os.path.join(image_folder, image_file)
        similar_images_list.append(image_path)

    display_single_image(query_image_path) # Display the downloaded image
    display_multiple_images(similar_images_list)

# Clean up the temporary file if it was created from a URL
if original_query_image_source.startswith('http://') or original_query_image_source.startswith('https://'):
    os.remove(query_image_path)